# Task 2: Analysis of Fake Data (Group 1 - Bennett, Alpert, and Goldstein's S)

In this notebook, we simulate fake annotation labels for two annotators across different prevalences of the label `"offensive"` ($N = 100$ samples per run).

We want to observe:
1. How does our group's metric (**Bennett's S**) behave as the prevalence of "offensive" changes from $0.0$ to $1.0$?
2. What happens to raw Percentage Agreement ($P_o$) vs Bennett's $S$?
3. Prepare the results and plots for our group presentation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# nice plot settings
plt.rcParams["figure.dpi"] = 120


In [ ]:
def calc_bennett_s(ann1, ann2, k=2):
    # compute raw percentage agreement
    po = np.mean(ann1 == ann2)
    # formula from Bennett et al. (1954): S = 1 - k*(1 - Po)/(k - 1)
    # for k=2 this simplifies to S = 2*Po - 1
    s = 1.0 - (k * (1.0 - po)) / (k - 1.0)
    return po, s


## 1. Simulating Fake Annotations Across Prevalences

We test prevalences from $0.0$ (no offensive tweets) to $1.0$ (all offensive tweets) in steps of $0.1$.
For each prevalence $p$, both annotators independently assign labels based on this probability ($N = 100$).


In [ ]:
# set seed so we get consistent numbers when rerunning
np.random.seed(42)

n_samples = 100
prevalences = np.linspace(0.0, 1.0, 11)  # 0.0, 0.1, 0.2, ..., 1.0

results = []

for p in prevalences:
    # draw random labels for annotator 1 and annotator 2
    ann1 = np.random.choice(["offensive", "not offensive"], size=n_samples, p=[p, 1.0 - p])
    ann2 = np.random.choice(["offensive", "not offensive"], size=n_samples, p=[p, 1.0 - p])
    
    po, s = calc_bennett_s(ann1, ann2)
    
    # count how many offensive each got just to see
    ann1_off = np.sum(ann1 == "offensive")
    ann2_off = np.sum(ann2 == "offensive")
    
    results.append({
        "prevalence": p,
        "ann1_offensive_count": ann1_off,
        "ann2_offensive_count": ann2_off,
        "percentage_agreement": po,
        "bennett_s": s
    })

df_sim = pd.DataFrame(results)
df_sim[["prevalence", "ann1_offensive_count", "ann2_offensive_count", "percentage_agreement", "bennett_s"]]


## 2. Plotting Bennett's S at Different Prevalences


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(df_sim["prevalence"], df_sim["percentage_agreement"], marker="s", linestyle="--", color="#3498db", label="Percentage Agreement ($P_o$)", linewidth=2)
plt.plot(df_sim["prevalence"], df_sim["bennett_s"], marker="o", color="#e67e22", label="Bennett's S ($S$)", linewidth=2.5)

# add reference line at S = 0
plt.axhline(0, color="gray", linestyle=":", alpha=0.7)

plt.xlabel("Prevalence of 'offensive' label", fontsize=11)
plt.ylabel("Score (0.0 to 1.0)", fontsize=11)
plt.title("Task 2: Bennett's S vs. Prevalence on Fake Data (N=100)", fontsize=13, fontweight="bold")
plt.xticks(prevalences)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=10)

plt.tight_layout()
plt.show()


## 3. Alternative Scenario: Annotators with Fixed High Agreement

What if the annotators are not just randomly guessing, but actually agree with each other ~90% of the time, while the true prevalence of offensive tweets changes?


In [ ]:
# simulate scenario where annotators actually have consistent agreement (~90%)
fixed_results = []

for p in prevalences:
    # generate true labels with prevalence p
    true_labels = np.random.choice(["offensive", "not offensive"], size=n_samples, p=[p, 1.0 - p])
    
    # annotators agree with ground truth 95% of the time (approx 90% mutual agreement)
    noise1 = np.random.rand(n_samples) < 0.05
    noise2 = np.random.rand(n_samples) < 0.05
    
    ann1 = np.where(noise1, np.where(true_labels == "offensive", "not offensive", "offensive"), true_labels)
    ann2 = np.where(noise2, np.where(true_labels == "offensive", "not offensive", "offensive"), true_labels)
    
    po, s = calc_bennett_s(ann1, ann2)
    fixed_results.append({
        "prevalence": p,
        "percentage_agreement": po,
        "bennett_s": s
    })

df_fixed = pd.DataFrame(fixed_results)

plt.figure(figsize=(9, 5))
plt.plot(df_fixed["prevalence"], df_fixed["percentage_agreement"], marker="s", linestyle="--", color="#2ecc71", label="PA (Consistent Annotators)", linewidth=2)
plt.plot(df_fixed["prevalence"], df_fixed["bennett_s"], marker="o", color="#9b59b6", label="Bennett's S (Consistent Annotators)", linewidth=2.5)
plt.xlabel("Prevalence of 'offensive'", fontsize=11)
plt.ylabel("Score", fontsize=11)
plt.title("Bennett's S with High-Agreement Annotators Across Prevalences", fontsize=13, fontweight="bold")
plt.xticks(prevalences)
plt.ylim(0, 1.1)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Summary & Findings for Presentation

### Key Insights:
1. **When annotators guess independently with probability $p$**:
   - At $p = 0.5$ (balanced), raw agreement is around $50\%$, so Bennett's $S = 2(0.5) - 1 = 0.0$.
   - As $p$ approaches $0.0$ or $1.0$ (extreme prevalence), raw agreement by chance alone naturally increases towards $100\%$ because both annotators mostly pick the same majority label.
   - Because Bennett's formula assumes a fixed uniform expected agreement ($P_e = 0.5$), Bennett's $S$ produces a **U-shaped curve** that rises at the extremes even though annotators are completely uncoordinated!
2. **When annotators have consistent accuracy (~90% agreement)**:
   - Bennett's $S$ stays almost completely flat across all prevalences ($S \approx 2(0.90) - 1 = 0.80$).
   - This illustrates that **Bennett's S is purely a function of raw percentage agreement $P_o$**, and is mathematically indifferent to empirical prevalence imbalance.
